# Question 7 - 121. Best Time to Buy and Sell Stock

You are given an array `prices` where `prices[i]` is the price of a given stock on the `i-th` day.

You want to maximize your profit by choosing a **single day** to buy one stock and choosing a **different day in the future** to sell that stock.

Return *the maximum profit you can achieve from this transaction*. If you cannot achieve any profit, return `0`.

**Example 1:**

    Input: prices = [7,1,5,3,6,4]
    Output: 5
    Explanation: Buy on day 2 (price = 1) and sell on day 5 (price = 6), profit = 6-1 = 5.
    Note that buying on day 2 and selling on day 1 is not allowed because you must buy before you sell.

**Example 2:**

    Input: prices = [7,6,4,3,1]
    Output: 0
    Explanation: In this case, no transactions are done and the max profit = 0.

**Constraints:**

- `1 <= prices.length <= 10^5`
- `0 <= prices[i] <= 10^4`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** One pass with a running minimum (greedy / 'best so far')

📘 **Revise first:** [Pattern C · running best-so-far](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (try every buy/sell pair) | O(n²) | O(1) |
| 2️⃣ Optimized (track cheapest price so far) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

You're given a stock's price for each day. You may **buy once** and **sell once**, and you must buy **before** you sell. What's the most money you can make? If every choice loses money, don't trade: the answer is `0`.

```
day:    0  1  2  3  4  5
price:  7  1  5  3  6  4
           ↑ buy      ↑ sell  → profit 6 − 1 = 5
```

**Everyday picture: looking back with hindsight.** Imagine walking through the days one at a time. On each day you ask yourself just one question:

> *"If I sold **today**, what's the best I could have done? That's today's price minus the **cheapest price I've seen so far**."*

You keep a note of the cheapest price so far, and a note of the best profit so far. That's the whole algorithm. The rest of this notebook shows why that's enough.

### Step 0 · Clarify before coding

🗣️ *"One buy and one sell, buy strictly before sell, and if no profit is possible I return 0, so the answer is never negative."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| How many trades? | **Exactly one** buy + one sell (or none). | Multiple trades is Question 8, a different strategy. |
| Same-day buy & sell? | Allowed in effect, profit 0. | Our answer is never below 0. |
| Prices only falling? | Return **0**. | Don't return a negative number. |
| Size? | Up to 10⁵ days. | O(n²) = 10¹⁰ checks, too slow. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** try **every** pair *(buy day, later sell day)*, compute the profit, and keep the best.

🗣️ *"The direct approach checks every buy day i and every later sell day j, computes prices[j] − prices[i], and keeps the maximum. It's obviously correct because it literally tries every allowed trade."*

**Why is it slow?** With n days there are about n²/2 pairs. For 100,000 days that's 5 billion pairs.

🗣️ *"That's O(n²). The waste: for each sell day, I'm re-scanning all earlier days to find the cheapest one, even though I already scanned most of them for the previous sell day."*

In [1]:
class SolutionBrute:
    def maxProfit(self, prices: list[int]) -> int:
        best = 0
        for i in range(len(prices)):              # buy day
            for j in range(i + 1, len(prices)):   # every later sell day
                best = max(best, prices[j] - prices[i])
        return best

### Step 2 · Optimize: the "aha" moment

Flip the brute force around. Instead of asking *"for this buy day, which sell day is best?"*, ask:

> 💡 **"If I sell on day j, the best buy day is simply the cheapest day *before* j."**

And "the cheapest price before day j" is something we can **keep updated as we walk**: one variable, `min_price`. When we move to the next day, the cheapest-so-far either stays the same or becomes today's price. We never need to look back.

So in one left-to-right pass:
1. `min_price = min(min_price, price)`: the best day to have bought, up to today.
2. `best = max(best, price − min_price)`: the profit if I sell today; keep the best ever.

(Updating `min_price` *before* computing today's profit means a same-day buy and sell gives profit 0, which is harmless.)

🗣️ *"For any sell day, the best buy is the minimum price before it. So I walk through the prices once, tracking the lowest price seen so far and the best profit so far. On each day I update the minimum, then check what I'd make by selling today against that minimum. That replaces the inner loop with a single variable: O(n) time and O(1) space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Try every pair | ❌ | O(n²). Rescans the past for every sell day. |
| Sort the prices | ❌ | Sorting destroys the **time order**, and "buy before sell" depends on order. |
| Global min and global max | ❌ | The max might come **before** the min (e.g. `[7, 1]`), so you'd be selling before buying. |
| Prefix-min array + suffix-max array | ⚠️ | Correct and O(n), but two extra arrays of size n. Unnecessary. |
| Kadane's on daily differences | ✅ (equivalent) | "Max subarray sum of price changes" is the same answer. A nice connection to mention (Kadane's section later). |
| **Running minimum, one pass** | ✅ **best** | O(n) time, O(1) memory, simple, and respects the order of days. |

**Takeaway pattern:** *"When the best choice for position j depends only on something about everything before j, keep that 'something' in a running variable."* (Concepts notebook, Pattern C.)

In [2]:
class Solution:
    def maxProfit(self, prices: list[int]) -> int:
        min_price = float("inf")    # cheapest price seen so far (best day to have bought)
        best = 0                    # best profit seen so far
        for price in prices:
            min_price = min(min_price, price)
            best = max(best, price - min_price)   # what if I sell today?
        return best

### Step 3 · Toy example walkthrough (tell it like a story)

`prices = [7, 1, 5, 3, 6, 4]`

| day | price | cheapest so far (`min_price`) | sell today → profit | best so far |
|---|---|---|---|---|
| 0 | 7 | 7 | 7 − 7 = 0 | 0 |
| 1 | 1 | **1** | 1 − 1 = 0 | 0 |
| 2 | 5 | 1 | 5 − 1 = 4 | 4 |
| 3 | 3 | 1 | 3 − 1 = 2 | 4 |
| 4 | 6 | 1 | 6 − 1 = **5** | **5** |
| 5 | 4 | 1 | 4 − 1 = 3 | 5 |

Answer **5** ✅ (buy at 1, sell at 6).

**Narrated:** "Day 0, the price is 7, the cheapest I've seen. Day 1, the price drops to 1, my new cheapest. On day 2, selling at 5 would earn 4, my best so far. Day 3 would only earn 2. Day 4, selling at 6 earns 5, a new best. Day 5 only earns 3. So the answer is 5."

**Tricky case:** `[2, 4, 1]`. On the last day the cheapest drops to 1, but there's no later day to sell, and `best` correctly stays at 2 (buy 2, sell 4). The new minimum doesn't erase the profit we already found.

In [3]:
cases = [
    ([7, 1, 5, 3, 6, 4], 5),
    ([7, 6, 4, 3, 1], 0),     # only falling -> no trade
    ([1], 0),                 # one day -> can't trade
    ([2, 4, 1], 2),           # new minimum after the best sale
    ([3, 3, 3], 0),
    ([1, 2], 1),
]
for prices, expected in cases:
    assert SolutionBrute().maxProfit(prices) == expected
    assert Solution().maxProfit(prices) == expected
    print(prices, "->", expected)

import random
for _ in range(500):
    p = [random.randint(0, 20) for _ in range(random.randint(1, 12))]
    assert Solution().maxProfit(p) == SolutionBrute().maxProfit(p), p
print("All tests passed ✅")

[7, 1, 5, 3, 6, 4] -> 5
[7, 6, 4, 3, 1] -> 0
[1] -> 0
[2, 4, 1] -> 2
[3, 3, 3] -> 0
[1, 2] -> 1
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Every pair | O(n²) | O(1) | For each day, look back at every earlier day. |
| **Running minimum** | **O(n)** | **O(1)** | Look at each day once, remembering two numbers. |

**Why O(n) time?** One loop, two `min`/`max` operations per day. Double the days, double the work.

**Why O(1) memory?** Just `min_price` and `best`, whatever the number of days.

**Edge cases to mention:** one day (0), strictly falling prices (0), flat prices (0), a new minimum that appears after the best sale.

**Likely follow-ups (this is a famous family of problems):**
- *Unlimited trades?* → Question 8: add up every price increase.
- *At most 2 trades / at most k trades?* → Questions 148/149: dynamic programming.
- *With a cooldown or a transaction fee?* → State-machine DP (holding / not holding).

---

#### 🗣️ Full interview script (about 60 seconds)

*"I can buy once and sell once later, and I want the maximum profit, or 0 if I can't profit.*

*Brute force checks every pair of buy and later sell days, which is O(n²), too slow for 10⁵ days.*

*The key observation is that if I sell on a given day, the best day to have bought is simply the cheapest day before it. So I walk through the prices once, keeping the minimum price so far and the best profit so far. Each day I update the minimum, then compute today's price minus that minimum and update the best profit.*

*That's a single pass with two variables, so O(n) time and O(1) space. It also respects the buy-before-sell order, because the minimum only ever includes earlier days."*